In [1]:
import sys
import os
import torch

# Federated Learning Experiments

This notebook has been updated to use the refactored testing approach from `test.py`.

In [2]:
# Import from core instead of run directly
# Run experiment using the test.py script
from core import load_configuration
import ray

# Ensure Ray is shut down to avoid conflicts
if ray.is_initialized():
    ray.shutdown()

In [3]:
# Run experiment using the test.py script
from test_file import run_gnn_experiments

# Define experiment parameters
data_loading_options = ["adjacency"]
model_types = ["GCN"]
dataset_name = "Cora"
clients_num = 2
beta_value = 10000
hop_value = 1

# Run the experiment
run_gnn_experiments(
    data_loading_options,
    model_types,
    dataset_name,
    clients_num,
    beta_value,
    hop_value
)

Running experiment: Cora_adjacency_GCN
DEVICE: cuda
Data loading option is adjacency
Model type is GCN


2025-04-11 17:21:33,720	INFO worker.py:1816 -- Started a local Ray instance.


data_loading_option: adjacency
Data loaded
Device is cuda
Cora()
2


(FLClient pid=3579536) 2025-04-11 17:21:42,384 - INFO - Epoch   0| Train Loss: 1.930| Train Accuracy: 0.197
(FLClient pid=3579536) 2025-04-11 17:21:42,394 - INFO - Epoch   0| Validation Loss: 1.933, Validation Accuracy: 0.183


Training done
Round 1
Train Loss: 1.933, Train Accuracy: 0.197
Train Loss: 1.942, Train Accuracy: 0.176
Validation Loss: 1.940, Validation Accuracy: 0.158
Validation Loss: 1.936, Validation Accuracy: 0.158

--- Model Parameter Comparison ---

Layer 0:
Server param shape: (16,)
Client param shapes: [(16,), (16,)]
✅ Parameters match!

Layer 1:
Server param shape: (16, 1433)
Client param shapes: [(16, 1433), (16, 1433)]
✅ Parameters match!

Layer 2:
Server param shape: (7,)
Client param shapes: [(7,), (7,)]
✅ Parameters match!

Layer 3:
Server param shape: (7, 16)
Client param shapes: [(7, 16), (7, 16)]
✅ Parameters match!

All model parameters are identical: True
The average client test results: 0.18599469580904912
The final global test results: 0.181
Round 1 is complete
The global test results: [0.181]
The client test results: [0.18599469580904912]
The average global test results: 0.181
The average client test results: 0.18599469580904912
The standad deviation global is: 0.0
The standad

(FLClient pid=3579538) 2025-04-11 17:21:42,462 - INFO - Epoch   0| Train Loss: 1.935| Train Accuracy: 0.176
(FLClient pid=3579538) 2025-04-11 17:21:42,482 - INFO - Epoch   0| Validation Loss: 1.942, Validation Accuracy: 0.116


Results saved to results_Cora_2_clients/Cora_adjacency_GCN/results_Cora_adjacency_GCN.txt



In [ ]:
import os
import ast # To safely evaluate string literal as dictionary
import pandas as pd
import logging # Using logging for better error reporting

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

def process_results_folder(main_folder_path):
    """
    Processes experiment results stored in subfolders of a main directory.

    Reads .txt files containing dictionary-like strings, parses them,
    and organizes the configuration and summary data into a Pandas DataFrame.

    Args:
        main_folder_path (str): The path to the main results folder
                                (e.g., 'Central_results2').

    Returns:
        pandas.DataFrame: A DataFrame containing the structured results,
                          with each row representing an experiment.
                          Returns an empty DataFrame if the folder is not found
                          or no valid results are processed.
    """
    all_results_data = [] # List to hold data for each valid experiment

    # --- Input Validation ---
    if not os.path.isdir(main_folder_path):
        logging.error(f"Folder not found at '{main_folder_path}'")
        return pd.DataFrame() # Return empty DataFrame

    logging.info(f"Starting processing for folder: '{main_folder_path}'")

    # --- Iterate Through Subfolders ---
    try:
        subfolder_names = os.listdir(main_folder_path)
    except OSError as e:
        logging.error(f"Could not list directory contents for '{main_folder_path}': {e}")
        return pd.DataFrame()

    for item_name in subfolder_names:
        subfolder_path = os.path.join(main_folder_path, item_name)

        # Check if it's actually a directory
        if not os.path.isdir(subfolder_path):
            logging.debug(f"Skipping non-directory item: '{item_name}'")
            continue

        logging.info(f"Processing subfolder: '{item_name}'")

        # --- Find the .txt File ---
        txt_file_path = None
        try:
            files_in_subfolder = os.listdir(subfolder_path)
            for filename in files_in_subfolder:
                if filename.endswith(".txt"):
                    # Assume only one relevant .txt file per subfolder
                    txt_file_path = os.path.join(subfolder_path, filename)
                    logging.debug(f"Found results file: '{filename}'")
                    break # Stop searching in this subfolder once found
            if not txt_file_path:
                 logging.warning(f"No .txt file found in subfolder '{item_name}'. Skipping.")
                 continue
        except OSError as e:
            logging.warning(f"Could not list files in subfolder '{item_name}': {e}. Skipping.")
            continue

        # --- Read and Parse File Content ---
        try:
            with open(txt_file_path, 'r', encoding='utf-8') as f:
                content = f.read()

            # Safely parse the string content into a Python dictionary
            # ast.literal_eval is safer than eval() for untrusted input
            data = ast.literal_eval(content)

            if not isinstance(data, dict):
                 logging.warning(f"Parsed content from '{txt_file_path}' is not a dictionary. Skipping.")
                 continue

        except FileNotFoundError:
            # This might happen in rare race conditions if the file is deleted
            # between listing and opening
            logging.warning(f"File '{txt_file_path}' found initially but could not be opened. Skipping.")
            continue
        except (SyntaxError, ValueError) as e:
            logging.warning(f"Error parsing content in file '{txt_file_path}': {e}. Skipping.")
            continue
        except Exception as e:
            logging.error(f"An unexpected error occurred reading/parsing '{txt_file_path}': {e}. Skipping.")
            continue

        # --- Extract Data ---
        # Use .get() for safer access in case keys are missing
        config = data.get('experiment_config', {})
        summary = data.get('summary', {})

        if not config or not summary:
             logging.warning(f"Missing 'experiment_config' or 'summary' key in '{txt_file_path}'. Skipping file.")
             continue

        # Create a flat dictionary for the DataFrame row
        # Start with configuration parameters
        row_data = config.copy()

        # Add summary results, prefixing keys if needed to avoid clashes (optional)
        # Example: prefix summary keys with 'summary_'
        # summary_prefixed = {f'summary_{k}': v for k, v in summary.items()}
        # row_data.update(summary_prefixed)

        # Or just update directly if keys are distinct
        row_data.update(summary)

        # Add an identifier for the experiment (using subfolder name)
        row_data['experiment_id'] = item_name
        # You could also add the filename if needed:
        # row_data['source_file'] = os.path.basename(txt_file_path)

        # Remove potentially large list/dict columns if not needed directly in the summary table
        # Adjust these based on what you want in the final table
        row_data.pop('global_results', None) # Remove list of all global results per round
        row_data.pop('client_results', None) # Remove list of all client results per round
        row_data.pop('rounds', None)         # Remove the detailed rounds data

        all_results_data.append(row_data)
        logging.debug(f"Successfully processed data for experiment '{item_name}'")

    # --- Create DataFrame ---
    if not all_results_data:
        logging.warning("No valid experiment data found to create a DataFrame.")
        return pd.DataFrame()

    try:
        results_df = pd.DataFrame(all_results_data)
        logging.info(f"Successfully created DataFrame with {len(results_df)} rows.")

        # Optional: Reorder columns for better readability
        id_col = ['experiment_id']
        config_cols = list(data.get('experiment_config', {}).keys())
        summary_cols = list(k for k in data.get('summary', {}).keys() if k not in ['global_results', 'client_results']) # Get summary keys added
        # Filter out columns that might not exist in all rows if parsing failed partially
        existing_cols = [col for col in id_col + config_cols + summary_cols if col in results_df.columns]
        results_df = results_df[existing_cols]

        # Optional: Set experiment_id as the DataFrame index
        # results_df.set_index('experiment_id', inplace=True)

    except Exception as e:
        logging.error(f"Failed to create DataFrame from processed data: {e}")
        return pd.DataFrame()


    return results_df

# --- Example Usage ---
# Note: Replace 'Central_results2' with the actual path to your folder.
# If running this script, make sure the folder exists relative to the script
# or provide an absolute path.

# Create dummy structure for demonstration if needed
def create_dummy_structure(base_dir="Central_results2_demo"):
    if not os.path.exists(base_dir):
        os.makedirs(base_dir)
    # Sample data similar to the user's example
    experiments_data = {
        "Pubmed_adjacency_GCN": {'experiment_config': {'device': 'cuda', 'data_loading_option': 'adjacency', 'model_type': 'GCN', 'dataset': 'Pubmed', 'num_clients': 10, 'beta': 1, 'hop': 1, 'fulltraining_flag': False}, 'rounds': [{'round': 1, 'global_result': 0.715, 'client_result': 0.724}, {'round': 2, 'global_result': 0.67, 'client_result': 0.698}], 'summary': {'global_results': [0.715, 0.67], 'client_results': [0.724, 0.698], 'average_global_result': 0.6925, 'average_client_result': 0.711, 'std_global': 0.0225, 'std_client': 0.013}},
        "Pubmed_diffusion_GCN": {'experiment_config': {'device': 'cuda', 'data_loading_option': 'diffusion', 'model_type': 'GCN', 'dataset': 'Pubmed', 'num_clients': 10, 'beta': 1, 'hop': 2, 'fulltraining_flag': False}, 'rounds': [], 'summary': {'average_global_result': 0.701, 'average_client_result': 0.715, 'std_global': 0.021, 'std_client': 0.012}},
        "Cora_full_SAGE": {'experiment_config': {'device': 'cpu', 'data_loading_option': 'full', 'model_type': 'SAGE', 'dataset': 'Cora', 'num_clients': 1, 'beta': 0, 'hop': 0, 'fulltraining_flag': True}, 'rounds': [], 'summary': {'average_global_result': 0.81, 'average_client_result': 0.81, 'std_global': 0.0, 'std_client': 0.0}},
        "Folder_With_Invalid_Txt": "This is not a dictionary string", # Test invalid content
        "Folder_With_No_Txt": {}, # Test missing txt file
        "Folder_With_Missing_Keys": {'experiment_config': {'device': 'cpu'}, 'summary': {}} # Test missing keys
    }
    logging.info(f"Creating dummy structure in '{base_dir}'...")
    for folder_name, data in experiments_data.items():
        subfolder_path = os.path.join(base_dir, folder_name)
        if not os.path.exists(subfolder_path):
            os.makedirs(subfolder_path)

        if folder_name == "Folder_With_No_Txt":
            # Create an empty file or a file with a different extension
            with open(os.path.join(subfolder_path, "other_file.log"), 'w') as f:
                f.write("Log data")
            continue # Don't create the specific .txt file

        # Create the results text file
        txt_filename = f"results_{folder_name}.txt"
        txt_filepath = os.path.join(subfolder_path, txt_filename)
        try:
            with open(txt_filepath, 'w', encoding='utf-8') as f:
                if isinstance(data, dict):
                    # Convert dict to string representation for the file
                    f.write(str(data))
                else:
                    # Write invalid data directly
                    f.write(data)
        except IOError as e:
            logging.error(f"Failed to write dummy file {txt_filepath}: {e}")
    logging.info("Dummy structure created.")


# --- Main execution block ---
if __name__ == "__main__":
    # Create dummy data for testing
    dummy_folder_path = "Central_results2_demo"
    create_dummy_structure(dummy_folder_path)

    # Process the results folder
    results_dataframe = process_results_folder(dummy_folder_path)

    # Display the results
    if not results_dataframe.empty:
        print("\n--- Processed Results DataFrame ---")
        # Configure pandas display options for better readability
        pd.set_option('display.max_columns', None) # Show all columns
        pd.set_option('display.width', 1000)       # Adjust width
        print(results_dataframe)
    else:
        print("\nNo DataFrame was generated. Check log messages for details.")

    # Clean up dummy structure (optional)
    # import shutil
    # try:
    #     shutil.rmtree(dummy_folder_path)
    #     logging.info(f"Cleaned up dummy directory: '{dummy_folder_path}'")
    # except OSError as e:
    #     logging.error(f"Error removing dummy directory '{dummy_folder_path}': {e}")


In [ ]:
results_folder = "Central_results2"
df = process_results_folder(results_folder)

In [ ]:
df

In [ ]:
results_folder2 = "Central_results3"
df2 = process_results_folder(results_folder2)
df2

In [ ]:
results_folder3 = "Central_results4"
df3 = process_results_folder(results_folder3)
df3